In [1]:
import pandas as pd

df = pd.concat([pd.read_json('../data/train.json'),
                pd.read_json('../data/test.json'),
                pd.read_json('../data/val.json')])

df = df[df.Query.str.contains('food|Food|item')]
df = df[['Query', 'SQL']].reset_index().drop(columns='index')

print(len(df))
df.head()

1428


,Query,SQL
0,How much did we spend on Food & drink This yea...,select sum(debit) from master_txn_table as T1...
1,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
2,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
3,What is the product name of the most frequentl...,SELECT Product_service FROM master_txn_table w...
4,Show the invoice number and the number of item...,"SELECT transaction_id , sum(quantity) FROM ma..."


In [4]:
df = pd.read_csv('../data/add_pair.csv',sep='/')
df.head()

,question,query
0,What food did I buy yesterday?,"SELECT items_info, vendor_name, final_cost, is..."
1,Give me total expenses for food on 20 June,SELECT SUM(final_cost) AS total_expenses FROM ...
2,Where did I buy hamburger from last 7 days,"SELECT DISTINCT vendor_name, vendor_address, i..."
3,What drinks did I buy today?,"SELECT items_info, vendor_name, final_cost, is..."
4,What was the total amount I spent yesterday?,SELECT SUM(final_cost) AS total_spent FROM rec...


In [5]:
from receipt_ai.databases.vectordb import ChromaVectorDB
from receipt_ai.config.config import settings

chromadb = ChromaVectorDB(collection_name=settings.CHROMA_COLLECTION_NAME)

In [4]:
from optimum.onnxruntime import ORTModelForFeatureExtraction
from transformers import AutoTokenizer
from pathlib import Path

SAVE_PATH = Path("../models_ckpt/onnx")

model_id = "sentence-transformers/all-MiniLM-L6-v2"

# load vanilla transformers and convert to onnx
model = ORTModelForFeatureExtraction.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# save onnx checkpoint and tokenizer
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

('../models_ckpt/onnx/tokenizer_config.json',
 '../models_ckpt/onnx/special_tokens_map.json',
 '../models_ckpt/onnx/vocab.txt',
 '../models_ckpt/onnx/added_tokens.json',
 '../models_ckpt/onnx/tokenizer.json')

In [6]:
for i, row in df.iterrows():
    chromadb.insert(row.question, {'sql':row.query}, "default")

In [3]:
from receipt_ai.models.embeddings import DefaultEmbeddingModel

emb_func = DefaultEmbeddingModel()
text = 'give me total expenses for food on 20 June'
chromadb.select(text, emb_func([text])[0],3)

{'ids': [['3e4e09f7-de35-41fb-96b6-a8a8ff111c1e',
   'd4811243-54f8-4e5e-bff0-f954474d78fd',
   '2dbe423d-6fa7-4e3f-b99a-a8b5892a517b']],
 'embeddings': None,
 'documents': [['Give me total expenses for food on 20 June',
   'List all receipts with subtotal above $20.',
   'Show all food purchases from McDonald’s.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'sql': "SELECT SUM(final_cost) AS total_expenses FROM receipt_ai.receipt_info_tb WHERE DATE(issued_date) = '2024-06-20' AND LOWER(items_info) LIKE REGEXP 'food|meal|dish|cuisine';"},
   {'sql': 'SELECT * FROM receipt_ai.receipt_info_tb WHERE subtotal > 20;'},
   {'sql': "SELECT * FROM receipt_ai.receipt_info_tb WHERE LOWER(vendor_name) LIKE '%mcdonald%' AND LOWER(items_info) REGEXP 'food|meal|burger|fries';"}]],
 'distances': [[-1.1920928955078125e-07,
   0.5106194019317627,
   0.5655871629714966]]}

In [8]:
from receipt_ai.prompts.prompt import UserReceiptQueryInsightPrompt
from receipt_ai.models.llm import GeminiLLM
from receipt_ai.tools.tool import ReceiptTools
from receipt_ai.utils.validate_request import database_info


tools = ReceiptTools()
llm = GeminiLLM(tools)
prompts = UserReceiptQueryInsightPrompt().template.substitute({"list_of_tools_name": ["get_or_store_data_from_query",
                                                                                     "get_ocr_inference"],
                                                                "database_info": database_info,
                                                                "receipt_image_path": "'../store_images/data.jpg'"})
history_chat = []
plain_prompts = UserReceiptQueryInsightPrompt()
prompts_dict = {"list_of_tools_name": ["get_or_store_data_from_query",
                                        "get_ocr_inference"],
                "database_info": database_info}

user_question = "Give me the detail info from the two latest receipt data that have been stored"

In [9]:
from receipt_ai.rags.rag import ReceiptRAG
from receipt_ai.databases.vectordb import ChromaVectorDB
from receipt_ai.config.config import settings

chromadb = ChromaVectorDB(collection_name=settings.CHROMA_COLLECTION_NAME)
rag_chain = ReceiptRAG(chromadb, llm, 3, ['sql'])

response, history_chat = rag_chain.invoke_and_save(user_question, history_chat, plain_prompts.template, prompts_dict)



        You are an expert at generating MySQL queries and summarizing information related to receipts.
        Your primary task is to create valid MySQL SELECT queries, along with the required parameters and any necessary reasoning, based on the user’s question or input.

        You must follow these rules before returning any response:

        1. You must strictly use only the existing function tools provided: ['get_or_store_data_from_query', 'get_ocr_inference'].

        2. When calling these tools, you must supply valid parameters using the correct data types exactly as defined by the tool specifications.

        3. If a receipt_image_path is provided, you must:
        a. Call the appropriate OCR tool to obtain the OCR inference.
        b. Place the OCR results in the `processed ocr format` section.
        c. After that, you must call the provided SQL tool to store the OCR inference into the database using an `INSERT INTO` query.
        Make sure you correctly extract and 

2025-11-20 06:05:59,087 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2025-11-20 06:05:59,175 - INFO - AFC is enabled with max remote calls: 10.


$$ success


2025-11-20 06:06:02,976 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.006601702741214207,
  content=Content(
    parts=[
      Part(
        text="""```json
{
    "id": [
        "391",
        "rcpt_food_034"
    ],
    "vendor_name": [
        "AUTHENTIC MEXICAN JOINT",
        "Starbucks"
    ],
    "vendor_address": [
        "900 KIRKWOOD AVE WEST HOLLYWOOD, CA",
        "122 5th Ave, NYC, NY"
    ],
    "items_info": [
        "[{\"item_name\": \"CHICKEN BURRITO\", \"item_cost\": \"8.79\", \"item_type\": \"meal\"}, {\"item_name\": \"KIDS MEAL - MAKE OWN\", \"item_cost\": \"4.99\", \"item_type\": \"meal\"}, {\"item_name\": \"LARGE DRINK\", \"item_cost\": \"2.19\", \"item_type\": \"drink\"}, {\"item_name\": \"DOMESTIC BEER\", \"item_cost\": \"4.99\", \"item_type\": \"alcohol\"}]",
        "[{\"item_name\":\"Cold Brew - Nitro Infused\",\"item_cost\":4.99,\"item_type\":\"beverage\"}]"
    ],
    "issued_date": [
        "2018-12-14 06:23:00",
        "202

In [10]:
response

'```json\n{\n    "id": [\n        "391",\n        "rcpt_food_034"\n    ],\n    "vendor_name": [\n        "AUTHENTIC MEXICAN JOINT",\n        "Starbucks"\n    ],\n    "vendor_address": [\n        "900 KIRKWOOD AVE WEST HOLLYWOOD, CA",\n        "122 5th Ave, NYC, NY"\n    ],\n    "items_info": [\n        "[{\\"item_name\\": \\"CHICKEN BURRITO\\", \\"item_cost\\": \\"8.79\\", \\"item_type\\": \\"meal\\"}, {\\"item_name\\": \\"KIDS MEAL - MAKE OWN\\", \\"item_cost\\": \\"4.99\\", \\"item_type\\": \\"meal\\"}, {\\"item_name\\": \\"LARGE DRINK\\", \\"item_cost\\": \\"2.19\\", \\"item_type\\": \\"drink\\"}, {\\"item_name\\": \\"DOMESTIC BEER\\", \\"item_cost\\": \\"4.99\\", \\"item_type\\": \\"alcohol\\"}]",\n        "[{\\"item_name\\":\\"Cold Brew - Nitro Infused\\",\\"item_cost\\":4.99,\\"item_type\\":\\"beverage\\"}]"\n    ],\n    "issued_date": [\n        "2018-12-14 06:23:00",\n        "2024-02-07 15:42:00"\n    ],\n    "subtotal": [\n        20.96,\n        4.99\n    ],\n    "tax_rate":

In [2]:
print(prompts)


        You are an expert at generating MySQL queries and summarizing information related to receipts.
        Your primary task is to create valid MySQL SELECT queries, along with the required parameters and any necessary reasoning, based on the user’s question or input.

        You must follow these rules before returning any response:

        1. You must strictly use only the existing function tools provided: ['get_or_store_data_from_query', 'get_ocr_inference'].

        2. When calling these tools, you must supply valid parameters using the correct data types exactly as defined by the tool specifications.

        3. If a receipt_image_path is provided, you must:
        a. Call the appropriate OCR tool to obtain the OCR inference.
        b. Place the OCR results in the `processed ocr format` section.
        c. After that, you must call the provided SQL tool to store the OCR inference into the database using an `INSERT INTO` query.
        Make sure you correctly extract and 

In [3]:
history_chat = []

response, history_chat = llm.generate_output('Give me the detail info from the receipt from the provided path and give confimation is it save to database or not and the sql query also', prompts, history_chat)

2025-11-16 21:48:14,802 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2025-11-16 21:48:14,825 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-11-16 21:48:14,988 - INFO - Going to convert document batch...
2025-11-16 21:48:14,988 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-16 21:48:14,993 - INFO - Loading plugin 'docling_defaults'
2025-11-16 21:48:14,996 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-16 21:48:15,000 - INFO - Loading plugin 'docling_defaults'
2025-11-16 21:48:15,009 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']


** sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.00256255897693336,
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'file_path': '../store_images/data.jpg'
          },
          name='get_ocr_inference'
        )
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>
)] create_time=None model_version='gemini-2.0-flash' prompt_feedback=None response_id='LeQZabKrOPu94-EP47TC2Qo' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=16,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=16
    ),
  ],
  prompt_token_count=1182,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1182
    ),
  ],
  total_token_count=1198
) automatic_function_calling_history=None parsed=None


2025-11-16 21:48:16,013 - INFO - Auto OCR model selected ocrmac.
2025-11-16 21:48:16,022 - INFO - Accelerator device: 'mps'
2025-11-16 21:48:18,487 - INFO - Accelerator device: 'mps'
2025-11-16 21:48:19,273 - INFO - Processing document data.jpg
2025-11-16 21:48:22,386 - INFO - Finished converting document data.jpg in 7.56 sec.
2025-11-16 21:48:22,424 - WARNING - Parameter `strict_text` has been deprecated and will be ignored.
2025-11-16 21:48:22,459 - INFO - AFC is enabled with max remote calls: 10.


@@ {'result': '<!-- image -->\n\nAUTHENTIC MEXICAN JOINT 900 KIRKWOOD AVE WEST HOLLYWOOD, CA\n\nHOST: MAURA 12/14/2018\n\nORDER: 391\n\n11:43 AM\n\nCHICKEN BURRITO KIDS MEAL - MAKE OWN LARGE DRINK DOMESTIC BEER $8.79 $4.99 $2.19 $4.99\n\nSUBTOTAL: $20.96 TAX: $1.15 VISA 4932 #XXXXXXXXX AUTHORIZE... BALANCE DUE $22.11\n\nLIKE US ON FACEBOOK TO GET SPECIAL OFFERS BY EMAIL'}


2025-11-16 21:48:27,382 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


-- sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.009466295205676755,
  content=Content(
    parts=[
      Part(
        text="""```json
{
    vendor_name: "AUTHENTIC MEXICAN JOINT",
    vendor_address: "900 KIRKWOOD AVE WEST HOLLYWOOD, CA",
    items_info: [
        {
            item_name: "CHICKEN BURRITO",
            item_cost: "8.79",
            item_type: "meal"
        },
        {
            item_name: "KIDS MEAL - MAKE OWN",
            item_cost: "4.99",
            item_type: "meal"
        },
        {
            item_name: "LARGE DRINK",
            item_cost: "2.19",
            item_type: "drink"
        },
        {
            item_name: "DOMESTIC BEER",
            item_cost: "4.99",
            item_type: "alcohol"
        }
    ],
    issued_date: "12/14/2018 11:43 AM",
    subtotal: "20.96",
    tax_rate: "1.15",
    additional_cost: [],
    final_cost: "22.11"
}
```

```tool_code
print(default_api.get_or_stor

2025-11-16 21:48:29,507 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2025-11-16 21:48:29,647 - INFO - AFC is enabled with max remote calls: 10.


** sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.010163001188143032,
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'is_select': 'False',
            'query': 'INSERT INTO receipt_info_tb (id, vendor_name, vendor_address, items_info, issued_date, subtotal, tax_rate, additional_cost, final_cost, currency) VALUES (\'391\', \'AUTHENTIC MEXICAN JOINT\', \'900 KIRKWOOD AVE WEST HOLLYWOOD, CA\', \'[{"item_name": "CHICKEN BURRITO", "item_cost": "8.79", "item_type": "meal"}, {"item_name": "KIDS MEAL - MAKE OWN", "item_cost": "4.99", "item_type": "meal"}, {"item_name": "LARGE DRINK", "item_cost": "2.19", "item_type": "drink"}, {"item_name": "DOMESTIC BEER", "item_cost": "4.99", "item_type": "alcohol"}]\', \'2018-12-14 11:43:00\', 20.96, 1.15, \'[]\', 22.11, \'$\')'
          },
          name='get_or_store_data_from_query'
        )
      ),
    ],
    role='model'
  ),
  finish_rea

2025-11-16 21:48:30,690 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"


-- sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.05843296732221331,
  content=Content(
    parts=[
      Part(
        text="""The data from the receipt image has been saved to the database.

```sql
SELECT * FROM receipt_info_tb WHERE id = '391';
```"""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>
)] create_time=None model_version='gemini-2.0-flash' prompt_feedback=None response_id='PeQZacOiLLaCg8UPq-SF8A4' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=35,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=35
    ),
  ],
  prompt_token_count=2969,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2969
    ),
  ],
  total_token_count=3004
) automatic_function_calling_history=[] parsed=None


In [4]:
response

'```json\n{\n"get_or_store_data_from_query_response": {\n"result": "Data stored successfully."\n}\n}\n```\n\nData stored successfully.\n```sql\nSELECT * FROM receipt_info_tb WHERE id = \'391\';\n```'

In [3]:
history_chat = []

response, history_chat = llm.generate_output('Give me the detail info from the receipt from the provided path as json, no need to call database', prompts, history_chat)

2025-11-16 14:56:48,900 - INFO - HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.0-flash:generateContent "HTTP/1.1 200 OK"
2025-11-16 14:56:49,059 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]


sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  avg_logprobs=-0.00026053207693621516,
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'file_path': '../store_images/data.jpg'
          },
          name='get_ocr_inference'
        )
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>
)] create_time=None model_version='gemini-2.0-flash' prompt_feedback=None response_id='v4MZaYXLO6urg8UPuuix2Qo' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=16,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=16
    ),
  ],
  prompt_token_count=1041,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1041
    ),
  ],
  total_token_count=1057
) automatic_function_calling_history=None parsed=None


2025-11-16 14:56:50,524 - INFO - Going to convert document batch...
2025-11-16 14:56:50,543 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-16 14:56:50,702 - INFO - Loading plugin 'docling_defaults'
2025-11-16 14:56:50,762 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-16 14:56:50,807 - INFO - Loading plugin 'docling_defaults'
2025-11-16 14:56:50,859 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-16 14:56:55,686 - INFO - Auto OCR model selected ocrmac.
2025-11-16 14:56:55,816 - INFO - Accelerator device: 'mps'
2025-11-16 14:57:01,040 - INFO - Accelerator device: 'mps'
2025-11-16 14:57:01,856 - INFO - Processing document data.jpg
2025-11-16 14:57:06,576 - INFO - Finished converting document data.jpg in 17.53 sec.
2025-11-16 14:57:06,676 - WARNING - Parameter `strict_text` has been deprecated and will be ignored.
2025-11-16 14:57:06,711 - INFO 

In [4]:
print(response)

```json
{
    "vendor_name": "AUTHENTIC MEXICAN JOINT",
    "vendor_address": "900 KIRKWOOD AVE WEST HOLLYWOOD, CA",
    "items_info": [
        {
            "item_name": "CHICKEN BURRITO",
            "item_cost": "8.79",
            "item_type": "food"
        },
        {
            "item_name": "KIDS MEAL - MAKE OWN",
            "item_cost": "4.99",
            "item_type": "food"
        },
        {
            "item_name": "LARGE DRINK",
            "item_cost": "2.19",
            "item_type": "drink"
        },
        {
            "item_name": "DOMESTIC BEER",
            "item_cost": "4.99",
            "item_type": "drink"
        }
    ],
    "issued_date": "2018-12-14 11:43:00",
    "subtotal": "20.96",
    "tax_rate": "0.055",
    "additional_cost": [],
    "final_cost": "22.11"
}
```



In [4]:
print(history_chat)

[Content(
  parts=[
    Part(
      text="""
        You are an expert at creating MySQL query and summarizing related to receipts. 
        Your task will be creating valid SQL query (one and only `select` type) along with the parameters and necessary analysis based on the corresponding user questions/inputs.
        You must follow this rules before returning the response:
        1. You strictly need to use the existing function tools that have been provided, namely ['get_data_from_query'] 
        2. When calling the corresponding tools, you need to provide the valid parameters which following the correct data type from the the defined tools
        3. You only have the knowledge to do reasoning about receipts that have been asked and stored by the user and can not perform query aside from `select` type.
           It also prohibited to answer unrelated questions. If the unrelated questions occured, you must answer apologetic statement.


        You must obey the output format und

In [2]:
tools.get_ocr_inference('../store_images/data.jpg')

2025-11-16 14:22:58,600 - INFO - detected formats: [<InputFormat.IMAGE: 'image'>]
2025-11-16 14:22:58,935 - INFO - Going to convert document batch...
2025-11-16 14:22:58,936 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 44ae89a68fc272bc7889292e9b5a1bad
2025-11-16 14:22:58,942 - INFO - Loading plugin 'docling_defaults'
2025-11-16 14:22:58,946 - INFO - Registered picture descriptions: ['vlm', 'api']
2025-11-16 14:22:58,951 - INFO - Loading plugin 'docling_defaults'
2025-11-16 14:22:58,960 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2025-11-16 14:23:00,746 - INFO - Auto OCR model selected ocrmac.
2025-11-16 14:23:00,757 - INFO - Accelerator device: 'mps'
2025-11-16 14:23:05,006 - INFO - Accelerator device: 'mps'
2025-11-16 14:23:05,869 - INFO - Processing document data.jpg
2025-11-16 14:23:08,877 - INFO - Finished converting document data.jpg in 10.28 sec.
2025-11-16 14:23:08,918 - WARNING - Parameter `str

{'result': '<!-- image -->\n\nAUTHENTIC MEXICAN JOINT 900 KIRKWOOD AVE WEST HOLLYWOOD, CA\n\nHOST: MAURA 12/14/2018\n\nORDER: 391\n\n11:43 AM\n\nCHICKEN BURRITO KIDS MEAL - MAKE OWN LARGE DRINK DOMESTIC BEER $8.79 $4.99 $2.19 $4.99\n\nSUBTOTAL: $20.96 TAX: $1.15 VISA 4932 #XXXXXXXXX AUTHORIZE... BALANCE DUE $22.11\n\nLIKE US ON FACEBOOK TO GET SPECIAL OFFERS BY EMAIL'}

In [4]:
from receipt_ai.databases.sqldb import MySqlDB

sqldb = MySqlDB()

sqldb.select(

"""
select count(*) from receipt_ai.receipt_info_tb;
"""
)

'{"count(*)":{"0":51}}'

In [2]:
import pandas as pd

pd.read_csv('../data/add_pair.csv', sep='|')

,question,query
0,What food did I buy yesterday?,"SELECT items_info, vendor_name, final_cost, is..."
1,Give me total expenses for food on 20 June,SELECT SUM(final_cost) AS total_expenses FROM ...
2,Where did I buy hamburger from last 7 days,"SELECT DISTINCT vendor_name, vendor_address, i..."
